# 🤖 RecruitAI — Enunciado de la Práctica
### Analizador automático de CVs con Google Gemini + Python + Streamlit

---

> **Nivel:** Básico-Intermedio &nbsp;|&nbsp; **Modalidad:** Guiada en clase

---

## 🎯 Objetivo

Construirás una aplicación web completa que:

1. **Lee CVs en formato PDF** y extrae su texto automáticamente
2. **Envía ese texto a la API de Google Gemini** con instrucciones precisas
3. **Recibe los datos estructurados en JSON** (nombre, experiencia, skills, score...)
4. **Muestra los resultados en una tabla interactiva** ordenada por puntuación
5. **Permite descargar el ranking en Excel** con un solo clic

Al terminar la práctica habrás aprendido a:
- Conectar un programa Python con una API de IA Generativa (LLM)
- Diseñar prompts de extracción estructurada
- Construir una interfaz web sin HTML usando Streamlit
- Organizar un proyecto Python en módulos reutilizables
- Proteger credenciales con variables de entorno

---
## 📋 Contexto del problema

Un equipo de selección de personal recibe decenas de CVs cada vez que publica una oferta. Leerlos uno a uno es lento, costoso y subjetivo.

Tu misión es construir una herramienta que automatice el **primer filtro de selección**: leer cada CV, extraer los datos relevantes, y puntuar qué tan bien encaja el candidato con la oferta. El equipo de RRHH seguirá tomando la decisión final, pero partirá de un ranking objetivo y documentado.

Esta es exactamente la arquitectura que utilizan hoy muchas empresas para escalar sus procesos de talent acquisition.

---
## 🗂️ Estructura del proyecto

Crea una carpeta llamada `recruitai/` en tu ordenador. Dentro tendrás que crear los siguientes archivos:

```
recruitai/
│
├── .env                  ← Tu API Key (nunca la compartas ni la subas a GitHub)
├── requirements.txt      ← Lista de librerías necesarias
│
├── pdf_loader.py         ← MÓDULO 1: Lee PDFs y devuelve texto
├── analyzer.py           ← MÓDULO 2: Llama a Gemini y devuelve JSON
└── app.py                ← MÓDULO 3: Interfaz web con Streamlit
```

Cada módulo tiene una **responsabilidad única y clara**. Esta separación hace el código más fácil de entender, probar y mantener. Es una buena práctica de ingeniería de software conocida como **principio de responsabilidad única**.

---
## ⚙️ Paso 0 — Configuración del entorno

### 0.1 Instala las dependencias

Crea el archivo `requirements.txt` con el siguiente contenido:

```
google-generativeai
pdfplumber
pypdf
pandas
python-dotenv
openpyxl
xlsxwriter
streamlit
```

Luego abre una terminal en la carpeta del proyecto y ejecuta:

```bash
py -m pip install -r requirements.txt
```

---

### 0.2 Obtén tu API Key de Google Gemini

1. Ve a [https://aistudio.google.com/](https://aistudio.google.com/) e inicia sesión con tu cuenta Google
2. Haz clic en **"Get API Key"** → **"Create API key"**
3. Copia la clave que se genera
4. Crea el archivo **`.env`** en la raíz de tu proyecto con este contenido:

```
GOOGLE_API_KEY=pega_aquí_tu_clave
```

> ⚠️ **Regla de oro:** La API Key es como una contraseña. **Nunca** la escribas directamente en el código Python. **Nunca** la subas a GitHub. Si la expones, cualquiera puede usarla y agotar tu cuota.

---

### 0.3 Verifica que todo está listo

Ejecuta la celda siguiente para comprobar que la API Key se lee correctamente:

In [ ]:
# VERIFICACIÓN DEL ENTORNO
# Ejecuta esta celda antes de empezar a construir los módulos

from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

if api_key:
    print(f"✅ API Key encontrada: {api_key[:8]}...{api_key[-4:]}")
    print("   El entorno está listo. Puedes continuar.")
else:
    print("❌ No se encontró la API Key.")
    print("   Comprueba que el archivo .env existe y tiene el formato correcto:")
    print("   GOOGLE_API_KEY=tu_clave_aquí")

---
## 📄 Módulo 1 — `pdf_loader.py`
### Leer el texto de un PDF

**Objetivo del módulo:** Recibir un archivo PDF y devolver su contenido como texto plano (`str`). Si no se puede leer, devolver `None`.

---

### ¿Por qué necesitamos esto?

La API de Gemini trabaja con texto, no con archivos PDF. Antes de poder analizar un CV, hay que convertirlo a texto plano. Las librerías `pdfplumber` y `pypdf` hacen exactamente eso.

---

### Lo que debes implementar

Una función llamada `get_pdf_text(pdf_path)` que:

**Intento 1 — método principal con `pdfplumber`:**
- Abre el PDF con `pdfplumber.open(pdf_path)`
- Itera por todas las páginas con `pdf.pages`
- En cada página llama a `pagina.extract_text()`
- Concatena el texto de todas las páginas

**Intento 2 — plan B con `pypdf` (si el intento 1 devuelve vacío):**
- Crea un `pypdf.PdfReader(pdf_path)`
- Itera por `reader.pages` y llama a `pagina.extract_text()`

**Devuelve** el texto si se extrajo algo, o `None` si ambos métodos fallaron.

---

### Pistas

- Usa bloques `try / except` para manejar errores: hay PDFs corruptos, con contraseña o con páginas vacías que pueden fallar
- Una página puede devolver `None` si está vacía — comprueba con `if texto_pagina:` antes de concatenar
- El parámetro `pdf_path` funciona igual con una ruta de texto (`str`) que con un objeto subido desde Streamlit
- Añade al inicio del archivo: `logging.getLogger("pdfminer").setLevel(logging.ERROR)` para silenciar avisos innecesarios

---

### Prueba tu módulo

Una vez escrito el archivo `pdf_loader.py`, ejecuta esta celda para verificar que funciona:

In [ ]:
# PRUEBA DEL MÓDULO 1
# Asegúrate de haber creado pdf_loader.py antes de ejecutar esta celda

from pdf_loader import get_pdf_text
import os

ruta_pdf = "tu_cv.pdf"  # <- Cambia esto por la ruta a cualquier PDF que tengas

if os.path.exists(ruta_pdf):
    texto = get_pdf_text(ruta_pdf)
    if texto:
        print(f"✅ PDF leído correctamente")
        print(f"   Caracteres extraídos: {len(texto)}")
        print(f"\n--- Primeros 600 caracteres ---")
        print(texto[:600])
    else:
        print("❌ No se pudo extraer texto.")
        print("   Posibles causas: PDF escaneado, PDF con contraseña, o archivo corrupto.")
else:
    print(f"⚠️  Archivo '{ruta_pdf}' no encontrado.")
    print("   Cambia la variable 'ruta_pdf' por la ruta correcta a tu PDF.")

### ✅ Criterios de superación del Módulo 1

- [ ] La función `get_pdf_text` existe en `pdf_loader.py`
- [ ] Acepta tanto rutas (`str`) como objetos de archivo de Streamlit
- [ ] Usa `pdfplumber` como método principal y `pypdf` como fallback
- [ ] Maneja errores con `try/except` sin romper el programa
- [ ] Devuelve `None` si no puede extraer texto
- [ ] La celda de prueba muestra texto extraído correctamente

---
## 🤖 Módulo 2 — `analyzer.py`
### Conectar con Gemini y extraer datos estructurados

**Objetivo del módulo:** Recibir el texto de un CV y el texto de una oferta de trabajo, enviárselos a Gemini con instrucciones precisas, y devolver un diccionario Python con todos los datos extraídos.

---

### Conceptos clave antes de empezar

**¿Qué es un prompt de extracción estructurada?**  
Es un conjunto de instrucciones que le damos al modelo para que, en lugar de responder con texto libre, devuelva los datos en un formato concreto y predecible. En nuestro caso, JSON.

**¿Por qué JSON?**  
Porque podemos convertirlo directamente en un diccionario Python con `json.loads()`, y de ahí a una fila de pandas, y de ahí a una fila de Excel. Todo el pipeline es automático.

**¿Qué es `temperature=0.0`?**  
La temperatura controla la creatividad del modelo. Con `0.0`, el modelo elige siempre la opción más probable, dando resultados más consistentes y reproducibles. Para extracción de datos, siempre usamos temperatura baja.

---

### Lo que debes implementar

Una función llamada `analyze_cv_with_gemini(cv_text, jd_text)` que:

**1. Cargue la API Key** desde el archivo `.env`:
```python
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
```

**2. Cree el cliente de Gemini:**
```python
from google import genai
client = genai.Client(api_key=api_key)
```

**3. Construya el prompt** que contenga:
- El **rol** del modelo (auditor de talento senior)
- Las **reglas de extracción** (qué contar, qué ignorar)
- Las **instrucciones de scoring** (cómo calcular la puntuación)
- El **texto de la oferta** y el **texto del CV** como inputs
- La **estructura exacta del JSON** de salida

**4. Llame a la API:**
```python
from google.genai import types

response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=tu_prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        temperature=0.0
    )
)
```

**5. Parsee y devuelva el JSON:**
```python
import json
datos = json.loads(response.text)
return datos
```

---

### El JSON que debe devolver el modelo

Instrúyele al modelo en el prompt para que devuelva **exactamente** esta estructura:

```json
{
    "nombre_candidato": "Nombre Completo",
    "categoria_profesional": "Junior",
    "anos_experiencia_total": 4.5,
    "anos_experiencia_data": 2.0,
    "num_empresas_trabajadas": 3,
    "ubicacion_actual": "Madrid",
    "sectores": ["Banca / Finanzas", "Tecnología"],
    "skills_tecnicas": ["Python", "SQL", "Power BI"],
    "certificaciones": ["AWS Cloud Practitioner"],
    "score_match": 76,
    "justificacion": "Perfil con buen dominio técnico pero escasa experiencia en banca."
}
```

---

### Reglas que debe seguir el modelo (inclúyelas en el prompt)

- **No contar como experiencia real:** Beca, Becario, Internship, Prácticas, Trainee, Student
- **`anos_experiencia_total`:** Años desde el primer trabajo real hasta el año actual (float, 1 decimal)
- **`anos_experiencia_data`:** Solo años en roles de Data Science / Analytics (float, 1 decimal)
- **`categoria_profesional`:** `"Becario"` si solo tiene prácticas, `"Junior"` (0-3 años), `"Senior"` (3-8 años), `"Manager"` (más de 8 años)
- **Si no encuentra un dato:** usar `null`, no inventar información
- **`score_match` (0-100):** Stack técnico 40%, sector 20%, años de experiencia 20%, ubicación 10%, certificaciones 10%

---

### Pistas

- Usa un f-string multilínea (`f"""..."""`). Así puedes insertar las variables directamente
- Incluye el esquema JSON dentro del prompt como ejemplo de output esperado — esto ayuda al modelo a entender exactamente qué quieres
- Envuelve toda la llamada a la API en `try/except` para capturar errores de red o de cuota
- A veces Gemini devuelve una lista con un solo elemento — comprueba con `isinstance(datos, list)` y devuelve `datos[0]` en ese caso

---

### Prueba tu módulo

In [ ]:
# PRUEBA DEL MÓDULO 2
# Asegúrate de haber creado analyzer.py antes de ejecutar esta celda

import json
from pdf_loader import get_pdf_text
from analyzer import analyze_cv_with_gemini

ruta_pdf = "tu_cv.pdf"  # <- Mismo PDF que antes

oferta_ejemplo = """
Posición: Data Scientist — Sector Financiero
Buscamos un perfil con al menos 3 años de experiencia en ciencia de datos.
Requisitos imprescindibles: Python, SQL, Machine Learning.
Valoramos: experiencia en sector bancario, certificaciones cloud (AWS/Azure/GCP),
conocimiento de herramientas de BI (Power BI, Tableau).
Ubicación: Madrid (modelo híbrido, 3 días presencial).
"""

if os.path.exists(ruta_pdf):
    print("Paso 1/2 — Leyendo PDF...")
    texto_cv = get_pdf_text(ruta_pdf)
    
    if texto_cv:
        print("Paso 2/2 — Enviando a Gemini... (puede tardar 10-20 segundos)")
        resultado = analyze_cv_with_gemini(texto_cv, oferta_ejemplo)
        
        if resultado:
            print("\n✅ ¡Análisis completado! Resultado recibido:\n")
            print(json.dumps(resultado, indent=2, ensure_ascii=False))
        else:
            print("❌ La función devolvió None. Revisa tu API Key y el código de analyzer.py")
    else:
        print("❌ No se pudo leer el PDF. Revisa el Módulo 1.")
else:
    print(f"⚠️  Archivo '{ruta_pdf}' no encontrado. Cambia la variable 'ruta_pdf'.")

### ✅ Criterios de superación del Módulo 2

- [ ] La función `analyze_cv_with_gemini` existe en `analyzer.py`
- [ ] La API Key se carga desde `.env`, nunca está escrita directamente en el código
- [ ] El prompt incluye: rol, reglas de extracción, inputs del CV y la oferta, y formato JSON de salida
- [ ] La llamada usa `response_mime_type="application/json"` y `temperature=0.0`
- [ ] La respuesta se parsea con `json.loads()` y se devuelve como diccionario
- [ ] Errores manejados con `try/except`, devuelve `None` si falla
- [ ] La celda de prueba muestra un JSON con todos los campos esperados

---

### 🔬 Ejercicio de exploración (opcional)

Una vez que el módulo funciona, prueba a modificar el prompt y observa cómo cambia la respuesta:

- Añade un campo nuevo al JSON: por ejemplo `"nivel_ingles": "Avanzado"`
- Cambia los pesos del scoring: dale más peso a las certificaciones
- Pide que la justificación sea más larga y detallada

Esta experimentación se llama **prompt engineering** y es una habilidad fundamental cuando se trabaja con LLMs.

---
## 🌐 Módulo 3 — `app.py`
### La interfaz web con Streamlit

**Objetivo del módulo:** Crear una aplicación web que permita subir PDFs, lanzar el análisis, ver los resultados en pantalla y descargarlos en Excel, sin escribir una sola línea de HTML.

---

### ¿Qué es Streamlit?

Streamlit convierte scripts Python en aplicaciones web interactivas. Cada vez que el usuario interactúa (sube un archivo, pulsa un botón), Streamlit **re-ejecuta el script completo de arriba a abajo**. Por eso los bloques `if st.button(...)` actúan como puntos de ramificación en el flujo.

Para lanzar la app, ejecuta en tu terminal:
```bash
streamlit run app.py
```

---

### Referencia rápida de componentes

| Componente | Código | Resultado |
|------------|--------|-----------|
| Título | `st.title("Texto")` | Encabezado grande |
| Subtítulo | `st.subheader("Texto")` | Encabezado mediano |
| Texto markdown | `st.markdown("**negrita**")` | Texto con formato |
| Separador | `st.markdown("---")` | Línea horizontal |
| Columnas | `col1, col2 = st.columns(2)` | Layout en columnas |
| Subir archivo | `st.file_uploader("Label", type=["pdf"])` | Botón de upload |
| Subir varios | `st.file_uploader(..., accept_multiple_files=True)` | Upload múltiple |
| Botón | `st.button("Texto", type="primary")` | Botón interactivo |
| Barra progreso | `barra = st.progress(0)` luego `barra.progress(0.5)` | Barra de progreso |
| Texto dinámico | `estado = st.empty()` luego `estado.text("msg")` | Texto actualizable |
| Tabla | `st.dataframe(df, use_container_width=True)` | Tabla interactiva |
| Descarga | `st.download_button(label, data, file_name, mime)` | Botón de descarga |
| Aviso | `st.warning("msg")` | Mensaje amarillo |
| Error | `st.error("msg")` | Mensaje rojo |
| Parar ejecución | `st.stop()` | Detiene el script |

---

### Lo que debes implementar

**1. Importaciones y configuración de página**
```python
st.set_page_config(page_title="RecruitAI", page_icon="🤖", layout="wide")
```

**2. Funciones auxiliares** (antes del código principal)

- `clean_name_for_file(nombre)` → Convierte `"José María de la Torre"` en `"Jose_Maria_de_la_Torre"`.  
  Pista: usa `unicodedata.normalize("NFKD", s).encode("ASCII", "ignore").decode("ASCII")` para quitar tildes, y `re.sub(r"[^a-zA-Z\s]", "", s)` para quitar caracteres especiales.

- `parse_list_data(campo)` → Si el campo es una lista, lo convierte en string separado por comas. Si ya es string, lo devuelve tal cual.

**3. Cabecera** con título y descripción breve.

**4. Sección de uploads en dos columnas:**
- Columna izquierda: uploader para la oferta de trabajo (un único PDF)
- Columna derecha: uploader para los CVs (múltiples PDFs permitidos)

**5. Botón de análisis** que al pulsarse:
- Valida que se han subido archivos — si no, muestra error y llama a `st.stop()`
- Lee el texto de la oferta con `get_pdf_text(jd_file)`
- Itera sobre cada CV: actualiza la barra de progreso, lee el texto, llama a `analyze_cv_with_gemini`, construye el diccionario `fila` y lo añade a la lista `resultados`
- Recuerda llamar a `cv_file.seek(0)` antes de pasarlo a `get_pdf_text`

**6. Mostrar resultados:**
- Convertir la lista `resultados` a DataFrame
- Ordenar por `"Score (0-100)"` descendente
- Mostrar con `st.dataframe()`

**7. Botón de descarga del Excel:**
- Usar `io.BytesIO()` y `pd.ExcelWriter` para generar el archivo en memoria
- Ofrecer descarga con `st.download_button()`

---

### Las columnas del DataFrame resultado

| Columna | Origen |
|---------|--------|
| `Fecha` | `datetime.now().strftime("%d-%m-%Y")` |
| `Archivo` | `cv_file.name` |
| `Nombre` | `datos["nombre_candidato"]` |
| `Categoría` | `datos["categoria_profesional"]` |
| `Score (0-100)` | `datos["score_match"]` |
| `Años Exp Total` | `datos["anos_experiencia_total"]` |
| `Años Exp Data` | `datos["anos_experiencia_data"]` |
| `Nº Empresas` | `datos["num_empresas_trabajadas"]` |
| `Ubicación` | `datos["ubicacion_actual"]` |
| `Sectores` | `parse_list_data(datos["sectores"])` |
| `Skills Técnicas` | `parse_list_data(datos["skills_tecnicas"])` |
| `Certificaciones` | `parse_list_data(datos["certificaciones"])` |
| `Justificación` | `datos["justificacion"]` |

---

### Pistas

- Usa `datos.get("campo", valor_por_defecto)` en lugar de `datos["campo"]` para evitar errores si falta algún campo
- Para el Excel en memoria: `buffer = io.BytesIO()` y luego `pd.ExcelWriter(buffer, engine="xlsxwriter")`
- El `mime` para un archivo Excel es `"application/vnd.ms-excel"`
- Si un CV no se puede leer, muestra `st.warning()` y continúa con el siguiente con `continue`

---

### ✅ Criterios de superación del Módulo 3

- [ ] `app.py` importa correctamente `get_pdf_text` y `analyze_cv_with_gemini`
- [ ] Hay un uploader para la oferta y otro múltiple para los CVs
- [ ] El botón valida que se han subido archivos antes de continuar
- [ ] Se procesa cada CV en bucle con barra de progreso visible
- [ ] Los resultados se muestran ordenados por score descendente
- [ ] Hay un botón para descargar el Excel
- [ ] Si un CV no se puede leer, muestra aviso y continúa sin romper la app
- [ ] La app funciona end-to-end: subir CVs → ver tabla → descargar Excel

---
## 🏆 Ejercicio de ampliación (para quien termine antes)

### Persistencia de datos entre sesiones

La versión básica pierde los resultados al cerrar la app. Modifica `app.py` para que:

1. Al arrancar, compruebe si existe `ranking_candidatos.xlsx`
2. Si existe, cargue los datos y muestre los candidatos ya analizados
3. Al analizar nuevos CVs, **combine** los nuevos resultados con los existentes
4. Guarde el archivo actualizado en disco tras cada análisis
5. **Evite duplicados**: si el nombre de archivo ya está en el Excel, no lo procese de nuevo

**Pista de implementación:**
```python
MASTER_FILE = "ranking_candidatos.xlsx"

if os.path.exists(MASTER_FILE):
    df_existente = pd.read_excel(MASTER_FILE)
    archivos_ya_procesados = df_existente["Archivo"].tolist()
else:
    df_existente = pd.DataFrame()
    archivos_ya_procesados = []

# Filtrar solo los nuevos
cvs_nuevos = [f for f in cv_files if f.name not in archivos_ya_procesados]

# Tras el análisis, combinar y guardar
df_combinado = pd.concat([df_existente, pd.DataFrame(resultados)], ignore_index=True)
df_combinado.to_excel(MASTER_FILE, index=False)
```

---
## 📦 Entregables

### Archivos obligatorios

| Archivo | Contenido mínimo |
|---------|------------------|
| `pdf_loader.py` | Función `get_pdf_text` con doble método de extracción |
| `analyzer.py` | Función `analyze_cv_with_gemini` con prompt completo |
| `app.py` | Aplicación Streamlit funcional end-to-end |
| `requirements.txt` | Lista de dependencias |
| `.env` | Archivo con la API Key (se verifica en clase, no se entrega) |

### Evidencias a mostrar en clase

1. **Captura de pantalla** de la aplicación Streamlit con al menos un CV analizado y la tabla de resultados visible
2. **Archivo Excel** descargado con el ranking de candidatos
3. **Respuesta a una pregunta** elegida al azar por el profesor:
   - ¿Qué hace el parámetro `response_mime_type="application/json"` en la llamada a la API?
   - ¿Por qué usamos `try/except` en `pdf_loader.py`?
   - ¿Qué pasaría si alguien sube un CV escaneado como imagen?
   - ¿Por qué la API Key está en `.env` y no directamente en `analyzer.py`?
   - ¿Qué ocurre en la aplicación Streamlit cuando el usuario pulsa el botón de análisis?

---
## 💬 Preguntas frecuentes

**¿Qué pasa si mi PDF está escaneado?**  
Los métodos de extracción solo funcionan con texto seleccionable. Si el PDF es una imagen, `get_pdf_text` devolverá `None`. La solución es OCR (reconocimiento óptico de caracteres), que queda fuera del alcance de esta práctica pero es una mejora natural del proyecto.

**¿Por qué el análisis tarda 10-20 segundos?**  
El tiempo corresponde a la llamada a la API de Gemini, que procesa el texto en los servidores de Google. Con CVs largos puede tardar un poco más.

**El modelo me devuelve scores ligeramente distintos cada vez para el mismo CV. ¿Es normal?**  
Con `temperature=0.0` el modelo debería ser determinista, pero pueden existir pequeñas variaciones. Si las diferencias son grandes (más de 10 puntos), revisa que estás usando `temperature=0.0`.

**¿Puedo añadir más campos al JSON?**  
Sí. Añádelos al prompt (en las instrucciones y en el esquema de salida) y al diccionario `fila` de `app.py`.

**¿Cómo se podría analizar contra múltiples ofertas simultáneamente?**  
En lugar de pasar un solo `jd_text`, se pasarían varios y se pediría al modelo un score por oferta. Así funciona exactamente el proyecto completo del profesor.

---
## 📌 Resumen: ¿qué aprenderás en esta práctica?

| Concepto | Dónde lo aplicarás |
|----------|--------------------|
| **LLM API** | `analyzer.py` — llamada a Gemini con `google-generativeai` |
| **Prompt Engineering** | Diseño del prompt con rol, reglas, inputs y formato de salida |
| **JSON estructurado** | Formato de salida del modelo, parseado con `json.loads()` |
| **Extracción de texto de PDF** | `pdf_loader.py` con `pdfplumber` y `pypdf` en cascada |
| **Variables de entorno** | `.env` + `python-dotenv` para proteger la API Key |
| **Interfaz web sin HTML** | `app.py` con Streamlit |
| **Manipulación de datos** | Pandas DataFrame + exportación a Excel |
| **Arquitectura modular** | Tres archivos con responsabilidades claramente separadas |
| **Manejo de errores** | `try/except` en los puntos críticos del pipeline |

---

*Enunciado de práctica — RecruitAI · Formación en IA Generativa aplicada · Nivel: Básico-Intermedio · Duración: 4h*